In [50]:
import pandas as pd
from pyprojroot import here

In [51]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [52]:
c2020 = pd.read_csv(RAW_DATA_DIR/"population/census_2020/conjunto_de_datos_iter_00CSV20.csv", low_memory=False)
c2010 = pd.read_csv(RAW_DATA_DIR/"population/census_2010/iter_00_cpv2010.csv")

In [53]:
c2020['CVEGEO'] = c2020['ENTIDAD'] * 1000 + c2020['MUN']
c2010['CVEGEO'] = c2010['entidad'] * 1000 + c2010['mun']

In [54]:
c2020 = c2020[(c2020["LOC"] > 0) & (c2020["LOC"] < 5000)].copy().reset_index(drop=True)
c2010 = c2010[(c2010["loc"] > 0) & (c2010["loc"] < 5000)].copy().reset_index(drop=True)

In [55]:
c2020_redux = c2020[['CVEGEO', 'POBTOT']].copy()
c2010_redux = c2010[['CVEGEO', 'pobtot']].copy()

In [56]:
c2020_redux.rename(columns={'POBTOT': '2020', 'anio': 'year'}, inplace=True)
c2010_redux.rename(columns={'pobtot': '2010', 'anio': 'year'}, inplace=True)

In [57]:
cdf2020 = c2020_redux.groupby('CVEGEO')[['2020']].sum()
cdf2010 = c2010_redux.groupby('CVEGEO')[['2010']].sum()

In [58]:
# c2020_redux.head(10)
c2010.head(10)

,entidad,nom_ent,mun,nom_mun,loc,nom_loc,longitud,latitud,altitud,pobtot,...,vph_tv,vph_refri,vph_lavad,vph_autom,vph_pc,vph_telef,vph_cel,vph_inter,tam_loc,CVEGEO
0,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1021746.0,215251.0,1885.0,722250,...,181062,173467,157607,114283,78107,104297,146684,55003,13.0,1001
1,1,Aguascalientes,1,Aguascalientes,94,Granja Adelita,1022225.0,215219.0,1903.0,14,...,*,*,*,*,*,*,*,*,1.0,1001
2,1,Aguascalientes,1,Aguascalientes,96,Agua Azul,1022127.0,215303.0,1863.0,37,...,11,10,9,3,1,0,6,1,1.0,1001
3,1,Aguascalientes,1,Aguascalientes,100,Rancho Alegre,1022222.0,215116.0,1880.0,10,...,*,*,*,*,*,*,*,*,1.0,1001
4,1,Aguascalientes,1,Aguascalientes,102,Los Arbolitos [Rancho],1022126.0,214649.0,1860.0,7,...,*,*,*,*,*,*,*,*,1.0,1001
5,1,Aguascalientes,1,Aguascalientes,104,Ardillas de Abajo (Las Ardillas),1021132.0,215643.0,1998.0,14,...,3,3,2,3,0,0,2,0,1.0,1001
6,1,Aguascalientes,1,Aguascalientes,106,Arellano,1021628.0,214803.0,1894.0,1382,...,250,228,208,135,37,85,190,23,4.0,1001
7,1,Aguascalientes,1,Aguascalientes,112,Bajío los Vázquez,1020726.0,214452.0,1972.0,55,...,11,10,8,9,0,1,7,0,1.0,1001
8,1,Aguascalientes,1,Aguascalientes,114,Residencial San Nicolás [Baños la Cantera],1022123.0,215104.0,1864.0,757,...,202,202,199,200,184,177,195,168,3.0,1001
9,1,Aguascalientes,1,Aguascalientes,120,Buenavista de Peñuelas,1021744.0,214303.0,1870.0,935,...,212,204,170,107,14,60,130,2,3.0,1001


In [59]:
cdf = pd.merge(cdf2020, cdf2010, on='CVEGEO', how='outer')
cdf2025 = cdf.reset_index().copy()

In [60]:
cdf

,2020,2010
CVEGEO,,
1001,948990,797010.0
1002,51536,45492.0
1003,58250,54136.0
1004,17000,15042.0
1005,129929,99590.0
...,...,...
32054,19446,18490.0
32055,31558,29395.0
32056,149607,138176.0


In [61]:
cdf2025['grw'] = (cdf2025['2020'] - cdf2025['2010']) / 10
# cdf2025['grw_8to24'] = (cdf2025['8to30_2020'] - cdf2025['8to30_2010']) / 10
# cdf2025['grw_unemp'] = (cdf2025['unemployed_2020'] - cdf2025['unemployed_2010']) / 10

for year in range(2011, 2020):
    cdf2025[str(year)] = cdf2025['2010'] + cdf2025['grw'] * (year - 2010)
    # cdf2025[f"8to30_{year}"] = cdf2025['8to30_2010'] + cdf2025['grw_8to24'] * (year - 2010)
    # cdf2025[f"unemployed_{year}"] = cdf2025['unemployed_2010'] + cdf2025['grw_unemp'] * (year - 2010)

for year in range(2021, 2026):
    cdf2025[str(year)] = cdf2025['2020'] + cdf2025['grw'] * (year - 2020)
    # cdf2025[f"8to30_{year}"] = cdf2025['8to30_2020'] + cdf2025['grw_8to24'] * (year - 2020)
    # cdf2025[f"unemployed_{year}"] = cdf2025['unemployed_2020'] + cdf2025['grw_unemp'] * (year - 2020)

cdf2025 = cdf2025.drop(columns=['grw']).reset_index(drop=True)
result = cdf2025.copy()
# id_cols = ['CVEGEO']  
# result = cdf2025.melt(id_vars=id_cols, var_name='metric_year', value_name='value')
# print(result.sample(10))

In [62]:
result.to_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv")

print(f" {len(result)} municipalities processed")
print(result.head(10))

 2469 municipalities processed
   CVEGEO    2020      2010      2011      2012      2013      2014      2015  \
0    1001  948990  797010.0  812208.0  827406.0  842604.0  857802.0  873000.0   
1    1002   51536   45492.0   46096.4   46700.8   47305.2   47909.6   48514.0   
2    1003   58250   54136.0   54547.4   54958.8   55370.2   55781.6   56193.0   
3    1004   17000   15042.0   15237.8   15433.6   15629.4   15825.2   16021.0   
4    1005  129929   99590.0  102623.9  105657.8  108691.7  111725.6  114759.5   
5    1006   47646   41862.0   42440.4   43018.8   43597.2   44175.6   44754.0   
6    1007   57369   49156.0   49977.3   50798.6   51619.9   52441.2   53262.5   
7    1008    9552    8443.0    8553.9    8664.8    8775.7    8886.6    8997.5   
8    1009   22485   19668.0   19949.7   20231.4   20513.1   20794.8   21076.5   
9    1010   20853   18828.0   19030.5   19233.0   19435.5   19638.0   19840.5   

       2016      2017      2018      2019      2021      2022      2023  \
0 